In [ ]:
# =========================================================
# INICIALIZAÇÃO
# =========================================================

In [ ]:
!pip -q install playwright pandas
!playwright install --with-deps chromium

In [ ]:
# =========================================================
# CONFIGURAÇÃO
# =========================================================

In [ ]:
import re
import time
import pandas as pd
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

GOOGLE_MAPS_URL = "https://www.google.com/maps/place/Museu+de+Arte+de+São+Paulo+Assis+Chateaubriand/@-23.561414,-46.6584568,17z/data=!3m1!4b1!4m6!3m5!1s0x94ce59ceb1eb771f:0xe904f6a669744da1!8m2!3d-23.561414!4d-46.6558819!16zL20vMDJfazR2!5m1!1e1?entry=ttu&g_ep=EgoyMDI2MDMwOC4wIKXMDSoASAFQAw%3D%3D"


In [ ]:
# =========================================================
# FUNÇÕES AUXILIARES
# =========================================================

In [ ]:
TOTAL_REVIEWS = 20
HEADLESS = True  # no Colab, normalmente deve ficar True


async def safe_text(locator, default=""): # Adicionado 'async'
    try:
        text = await locator.inner_text(timeout=3000) # Adicionado 'await'
        return text.strip()
    except:
        return default

def extract_number_from_text(text):
    if not text:
        return None
    match = re.search(r"(\d+)", text.replace(".", ""))
    return int(match.group(1)) if match else None

async def open_reviews_modal(page): # Adicionado 'async'
    """
    Tenta localizar e clicar no botão/área que abre a janela de reviews.
    Como o Google Maps muda com frequência, usamos algumas tentativas.
    """
    candidate_selectors = [
        'button[jsaction*="pane.reviewChart.moreReviews"]',
        'button[aria-label*="reviews"]',
        'button[aria-label*="Reviews"]',
        'button[aria-label*="avaliações"]',
        'button[aria-label*="Avaliações"]',
        'div[role="button"][aria-label*="reviews"]',
        'div[role="button"][aria-label*="avaliações"]',
    ]

    for selector in candidate_selectors:
        try:
            locator = page.locator(selector).first
            if await locator.count() > 0: # Adicionado 'await'
                await locator.click(timeout=5000) # Adicionado 'await'
                await page.wait_for_timeout(3000) # Adicionado 'await'
                return True
        except:
            pass

    # tentativa alternativa: clicar em algum texto visível contendo reviews/avaliações
    fallback_texts = ["reviews", "Reviews", "avaliações", "Avaliações"]
    for txt in fallback_texts:
        try:
            await page.get_by_text(txt, exact=False).first.click(timeout=5000) # Adicionado 'await'
            await page.wait_for_timeout(3000) # Adicionado 'await'
            return True
        except:
            pass

    return False

async def find_scrollable_reviews_container(page): # Adicionado 'async'
    """
    Localiza a área rolável da lista de reviews.
    """
    candidate_selectors = [
        'div[role="main"] div[aria-label*="reviews"]',
        'div[role="main"] div[aria-label*="Reviews"]',
        'div[role="main"] div[aria-label*="avaliações"]',
        'div[role="feed"]',
        'div.m6QErb[tabindex="0"]',
        'div.m6QErb[tabindex="-1"]',
        'div[aria-label][tabindex="0"]',
    ]

    for selector in candidate_selectors:
        try:
            locator = page.locator(selector).last
            if await locator.count() > 0: # Adicionado 'await'
                await locator.wait_for(timeout=3000) # Adicionado 'await'
                return locator
        except:
            pass

    return None

async def scroll_until_enough_reviews(page, scrollable, target_count=20, max_attempts=30): # Adicionado 'async'
    """
    Faz scroll na área de reviews até carregar reviews suficientes.
    """
    previous_count = 0

    for attempt in range(max_attempts):
        review_cards = page.locator('div.jftiEf, div[data-review-id], div.MyEned')
        current_count = await review_cards.count() # Adicionado 'await'

        print(f"Tentativa {attempt+1}: {current_count} reviews carregadas")

        if current_count >= target_count:
            break

        try:
            await scrollable.evaluate("(el) => { el.scrollTop = el.scrollHeight; }") # Adicionado 'await'
        except:
            await page.mouse.wheel(0, 4000) # Adicionado 'await'

        await page.wait_for_timeout(2500) # Adicionado 'await'

        if current_count == previous_count:
            # pequena insistência extra
            try:
                await scrollable.evaluate("(el) => { el.scrollTop = el.scrollHeight; }") # Adicionado 'await'
            except:
                pass
            await page.wait_for_timeout(2500) # Adicionado 'await'

        previous_count = current_count

async def expand_truncated_reviews(page): # Adicionado 'async'
    """
    Clica nos botões 'Mais' / 'More' para expandir textos truncados.
    """
    more_selectors = [
        'button.w8nwRe',
        'button[aria-label="More"]',
        'button[aria-label="Mais"]'
    ]

    for selector in more_selectors:
        try:
            buttons = page.locator(selector)
            count = await buttons.count() # Adicionado 'await'
            for i in range(count):
                try:
                    await buttons.nth(i).click(timeout=1000) # Adicionado 'await'
                    await page.wait_for_timeout(200) # Adicionado 'await'
                except:
                    pass
        except:
            pass

async def collect_reviews(page, limit=20): # Adicionado 'async'
    """
    Extrai dados das reviews já carregadas na tela.
    """
    data = []

    possible_cards = page.locator('div.jftiEf, div[data-review-id], div.MyEned')
    total_cards = await possible_cards.count() # Adicionado 'await'

    print(f"Total de cards localizados: {total_cards}")

    for i in range(min(total_cards, limit)):
        card = possible_cards.nth(i)

        # nome do autor
        author = ""
        for sel in ['div.d4r55', 'div[class*="d4r55"]', 'button div:first-child']:
            try:
                loc = card.locator(sel).first
                if await loc.count() > 0: # Adicionado 'await'
                    author = await safe_text(loc) # Adicionado 'await'
                    if author:
                        break
            except:
                pass

        # nota
        rating = None
        try:
            star_element = card.locator('span[aria-label*="star"], span[aria-label*="estrela"]').first
            star_text = await star_element.get_attribute("aria-label", timeout=2000) or "" # Adicionado 'await'
            rating = extract_number_from_text(star_text)
        except:
            pass

        # data da review
        review_date = ""
        for sel in ['span.rsqaWe', 'span[class*="rsqaWe"]']:
            try:
                loc = card.locator(sel).first
                if await loc.count() > 0: # Adicionado 'await'
                    review_date = await safe_text(loc) # Adicionado 'await'
                    if review_date:
                        break
            except:
                pass

        # texto da review
        review_text = ""
        for sel in [
            'span.wiI7pd',
            'div.MyEned span',
            'span[class*="wiI7pd"]',
            'div[jslog] span'
        ]:
            try:
                loc = card.locator(sel).first
                if await loc.count() > 0: # Adicionado 'await'
                    review_text = await safe_text(loc) # Adicionado 'await'
                    if review_text:
                        break
            except:
                pass

        # quantidade de contribuições do perfil
        contributions = None
        try:
            contrib_loc = card.locator('span.RfnDt').first
            contrib_text = await safe_text(contrib_loc) # Adicionado 'await'
            contributions = extract_number_from_text(contrib_text)
        except:
            pass

        data.append({
            "autor": author,
            "nota": rating,
            "data_review": review_date,
            "texto": review_text,
            "contribuicoes_usuario": contributions
        })

    return data

# =========================================================
# SCRAPER PRINCIPAL
# =========================================================
async def scrape_google_maps_reviews(url, total_reviews=20, headless=True): # Adicionado 'async'
    async with async_playwright() as p: # Alterado para 'async_playwright' e 'async with'
        browser = await p.chromium.launch(
            headless=headless,
            args=[
                "--disable-blink-features=AutomationControlled",
                "--no-sandbox",
                "--disable-dev-shm-usage"
            ]
        )

        context = await browser.new_context(
            locale="pt-BR",
            viewport={"width": 1400, "height": 2200},
            user_agent=(
                "Mozilla/5.0 (X11; Linux x86_64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/122.0.0.0 Safari/537.36"
            )
        )

        page = await context.new_page()

        print("Abrindo página...")
        await page.goto(url, wait_until="networkidle", timeout=120000) # Timeout aumentado para 120s
        await page.wait_for_timeout(5000)

        # Aceitar cookies, se aparecer
        cookie_texts = ["Aceitar tudo", "Accept all", "I agree"]
        for txt in cookie_texts:
            try:
                await page.get_by_text(txt, exact=False).click(timeout=3000)
                await page.wait_for_timeout(2000)
                break
            except:
                pass

        print("Tentando abrir modal de reviews...")
        opened = await open_reviews_modal(page)
        if not opened:
            await browser.close()
            raise Exception("Não consegui abrir a seção de reviews. Verifique a URL ou ajuste os seletores.")

        print("Localizando container rolável...")
        scrollable = await find_scrollable_reviews_container(page)
        if not scrollable:
            await browser.close()
            raise Exception("Não encontrei a área rolável de reviews.")

        print("Rolando a lista até carregar reviews suficientes...")
        await scroll_until_enough_reviews(page, scrollable, target_count=total_reviews, max_attempts=35)

        print("Expandindo textos truncados...")
        await expand_truncated_reviews(page)

        print("Coletando dados...")
        reviews = await collect_reviews(page, limit=total_reviews)

        await browser.close()
        return reviews



In [ ]:
# =========================================================
# EXECUÇÃO
# =========================================================
# O Colab já executa um loop de asyncio, então usamos asyncio.run para chamar a função assíncrona.
reviews_data = await scrape_google_maps_reviews(
    url=GOOGLE_MAPS_URL,
    total_reviews=TOTAL_REVIEWS,
    headless=HEADLESS
)

df_reviews = pd.DataFrame(reviews_data)

print(f"\nTotal extraído: {len(df_reviews)} reviews")
display(df_reviews.head(20))

In [ ]:
# =========================================================
# SALVAR EM CSV
# =========================================================

In [ ]:
df_reviews.to_csv("google_maps_reviews.csv", index=False, encoding="utf-8-sig")
print("Arquivo salvo como google_maps_reviews.csv")

In [ ]:
# =========================================================
# DOWNLOAD DO ARQUIVO NO COLAB
# =========================================================

In [ ]:
from google.colab import files
files.download("google_maps_reviews.csv")